# Factorial (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Factorial_cpp.ipynb)

Chapter 14 — Recursion. Runnable notebook of `Factorial`: the algorithm in C++, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Factorial_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Factorial_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Names.** There are no arrays in this chapter: variables and parameters have the same names as in the pseudo-code. The stack and the counters are global variables, and the trace is switched on with `trace = true`.

## Factorial

**Input:** a natural number $n$. **Output:** $n!$, with $0! = 1$. $S(n) = 2n + 2 = \Theta(n)$ steps; $n + 1$ calls, depth $n + 1$ (the tree is a path).

Pseudo-code: section 14.4 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [1]:
%%writefile factorial.cpp
#include <cstdarg>
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];      // cnt[r] = number of executions of line r (numbering of the lecture notes)
long long calls = 0;    // number of calls = nodes of the tree of recursive calls
int maxDepth = 0;       // depth of the recursion = maximum height of the stack
long long mults = 0;    // multiplications (FastPower)
long long moves = 0;    // moves (Hanoi)
int ev = 0;             // counter of the events of the trace
int stk[1000];          // the call stack: the arguments of the active calls (at most 1000)
int sp = 0;             // number of active calls
long long seen[64];     // seen[x] = number of calls with argument x
bool trace = false;     // true: print the trace of the execution

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
    for (int k = 0; k < 64; k++) seen[k] = 0;
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0; sp = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Line | Executions\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Total steps S = %lld\n", Total());
}

string Str(const char* fmt, ...)       // like sprintf, but returns a string
{
    char buf[256];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return string(buf);
}

void Push(int x)                       // a call starts: its frame goes on top of the stack
{
    calls++;
    stk[sp++] = x;
    if (sp > maxDepth) maxDepth = sp;
    if (x >= 0 && x < 64) seen[x]++;
}

void Pop()                             // the call returns: its frame leaves the stack
{
    sp--;
}

string StackStr()                      // "[4, 3, 2]"
{
    string s = "[";
    for (int k = 0; k < sp; k++)
    {
        if (k > 0) s += ", ";
        s += to_string(stk[k]);
    }
    return s + "]";
}

void Event(const string& text)         // numbered line of the trace, with the stack
{
    ev++;
    printf("%2d. %s   stack: %s\n", ev, text.c_str(), StackStr().c_str());
}

void Say(const string& call, const string& action)   // line of the trace: the call, indented one level for every active call
{
    printf("%-28s| %s\n", (string(2 * (sp - 1), ' ') + call).c_str(), action.c_str());
}

// Factorial(n) correct only up to n = 20 (in C++ and Java 64 bits overflow beyond)
unsigned long long Factorial(int n)
{
    Push(n); cnt[2]++;
    if (n == 0)
    {
        cnt[3]++;
        if (trace) Event("Factorial(0): n = 0 => base case");
        Pop();
        if (trace) Event("Factorial(0) returns 1");
        return 1;
    }
    if (trace) Event(Str("Factorial(%d): %d != 0 => must compute %d * Factorial(%d) and waits", n, n, n, n - 1));
    cnt[5]++;
    unsigned long long f = Factorial(n - 1);
    unsigned long long r = n * f;
    Pop();
    if (trace) Event(Str("Factorial(%d) returns %d * %llu = %llu", n, n, f, r));
    return r;
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Example of the lecture notes: Factorial(4), with the stack
    int n = 4;
    Reset();
    trace = true;
    unsigned long long r = Factorial(n);
    trace = false;
    printf("Factorial(%d) = %llu\n", n, r);
    printf("Calls = %lld (nodes of the tree), depth of the recursion = %d\n", calls, maxDepth);
    PrintSteps({2, 3, 5});
    printf("S(n) = 2n + 2 = %d\n", 2 * n + 2);

    // Calls, depth and steps as n varies
    printf("\n");
    printf("  n |                   n! |    calls |       depth | S(n) | 2n + 2\n");
    int values[] = {0, 1, 2, 3, 4, 5, 10, 15, 20};
    for (int v : values)
    {
        Reset();
        unsigned long long f = Factorial(v);
        printf("%3d | %20llu | %8lld | %11d | %4lld | %d\n", v, f, calls, maxDepth, Total(), 2 * v + 2);
    }
    return 0;
}
#endif

Writing factorial.cpp


In [2]:
!g++ -std=c++17 -O2 -o factorial factorial.cpp && ./factorial

 1. Factorial(4): 4 != 0 => must compute 4 * Factorial(3) and waits   stack: [4]
 2. Factorial(3): 3 != 0 => must compute 3 * Factorial(2) and waits   stack: [4, 3]
 3. Factorial(2): 2 != 0 => must compute 2 * Factorial(1) and waits   stack: [4, 3, 2]
 4. Factorial(1): 1 != 0 => must compute 1 * Factorial(0) and waits   stack: [4, 3, 2, 1]
 5. Factorial(0): n = 0 => base case   stack: [4, 3, 2, 1, 0]
 6. Factorial(0) returns 1   stack: [4, 3, 2, 1]
 7. Factorial(1) returns 1 * 1 = 1   stack: [4, 3, 2]
 8. Factorial(2) returns 2 * 1 = 2   stack: [4, 3]
 9. Factorial(3) returns 3 * 2 = 6   stack: [4]
10. Factorial(4) returns 4 * 6 = 24   stack: []
Factorial(4) = 24
Calls = 5 (nodes of the tree), depth of the recursion = 5
Line | Executions
   2 |          5
   3 |          1
   5 |          4
Total steps S = 10
S(n) = 2n + 2 = 10

  n |                   n! |    calls |       depth | S(n) | 2n + 2
  0 |                    1 |        1 |           1 |    2 | 2
  1 |                    1 |

**Try it yourself.** Change $n$ and watch the stack grow and then empty.

In [3]:
%%writefile prova_factorial.cpp
#define SOLO_LIBRERIA   // do not run the main of factorial.cpp
#include "factorial.cpp"

int main()
{
    int n = 6;        // <-- change n (20 at most)
    Reset();
    trace = true;
    unsigned long long r = Factorial(n);
    trace = false;
    printf("Factorial(%d) = %llu, calls = %lld, depth = %d, S = %lld\n", n, r, calls, maxDepth, Total());
    return 0;
}

Writing prova_factorial.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_factorial prova_factorial.cpp && ./prova_factorial

 1. Factorial(6): 6 != 0 => must compute 6 * Factorial(5) and waits   stack: [6]
 2. Factorial(5): 5 != 0 => must compute 5 * Factorial(4) and waits   stack: [6, 5]
 3. Factorial(4): 4 != 0 => must compute 4 * Factorial(3) and waits   stack: [6, 5, 4]
 4. Factorial(3): 3 != 0 => must compute 3 * Factorial(2) and waits   stack: [6, 5, 4, 3]
 5. Factorial(2): 2 != 0 => must compute 2 * Factorial(1) and waits   stack: [6, 5, 4, 3, 2]
 6. Factorial(1): 1 != 0 => must compute 1 * Factorial(0) and waits   stack: [6, 5, 4, 3, 2, 1]
 7. Factorial(0): n = 0 => base case   stack: [6, 5, 4, 3, 2, 1, 0]
 8. Factorial(0) returns 1   stack: [6, 5, 4, 3, 2, 1]
 9. Factorial(1) returns 1 * 1 = 1   stack: [6, 5, 4, 3, 2]
10. Factorial(2) returns 2 * 1 = 2   stack: [6, 5, 4, 3]
11. Factorial(3) returns 3 * 2 = 6   stack: [6, 5, 4]
12. Factorial(4) returns 4 * 6 = 24   stack: [6, 5]
13. Factorial(5) returns 5 * 24 = 120   stack: [6]
14. Factorial(6) returns 6 * 120 = 720   stack: []
Factorial(6) = 720, c

---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).